### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

    . Tracking agents behavior with logging, analytics and debugging.
    . Transforming prompts, tool selection and output formatting.
    . Adding retries,fallbacks and early termination logic.
    . Applying rate limits, guardrails, and PIL detection.

In [9]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

### Summarization Middleware

Automatically summarize conversation history when approaching token limits, preserving recent message while compressing older context. Summarization is useful for the following

    . Long-running conversation that exceed context windows.
    . Multi-turn dialogues with extension history.
    . Application where preserving full conversation context matters.

In [10]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Messagebased summarization
agent=create_agent(
    model="gpt-4o-mini",
    checkpointer=MemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [1]:
### Run with thread id
config={"configurable":{"thread_id":"test-1"}}

In [ ]:
# Alternative test data
questions = [
    "what is 2+2?",
    "what is 10*5?",
    "what is 100/4?",
    "what is 15-7?",
    "what is 3*3?",
    "what is 4*4?", 
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

### Token Size

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

@tool
def search_hotels(city: str) -> str:
    """Search hotels - return long response to use more tokens."""
    return f"""hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa,pool,gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""
    
agent=create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[search_hotels],
    checkpointer=MemorySaver()
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("token",550),
            keep=("token",200),
        )
    ]
)

config = {"configurable": {"thread_id":"test-1"}}

# token counter(approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content))for m in messages)
    return total_chars // 4   # 4 chars = 1 token